# 05. Mechanistic Interpretability: Probing, Logit Lens, Attribution Patching, and SAE

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Alain & Bengio (2016 *Understanding intermediate layers using linear classifier probes*); Hewitt & Liang (2019 *Designing and Interpreting Probes with Control Tasks*); nostalgebraist (2020 *interpreting gpt: the logit lens*); Belrose et al. (2023 *Eliciting Latent Knowledge from Quirky Language Models: Tuned Lens*); Syed et al. (2023 *Attribution Patching: Approximating Activation Patching*); Bricken et al. (2023 *Towards Monosemanticity: Decomposing Language Models With Dictionary Learning*); Gao et al. (2024 *Scaling and evaluating sparse autoencoders: Top-K SAEs*); Rajamanoharan et al. (2024 *JumpReLU: A Simple and Effective Threshold for SAEs*).
- **Purpose:** Reverse-engineer neural representations using the Logit Lens, fast first-order Attribution Patching (AtP), and overcomplete dictionary learning via Top-K and JumpReLU Sparse Autoencoders.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. XAI Data Attribution Influence Functions TracIn TRAK](04_xai_data_attribution_influence_functions_tracin_and_trak.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. AI Brain Surgery Knowledge Editing ROME MEMIT](06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Avoid deep non-linear probing architectures that memorize labels rather than measuring linearly accessible model representations; never use Attribution Patching as the sole ground truth across large non-linear perturbations without checking exact activation patching.

## Mathematical Foundations & Governing Equations

1. **Logit Lens**: $\text{logits}_l = h_l W_U + b_U, \quad p_l = \text{softmax}(\text{logits}_l)$
2. **Attribution Patching (AtP)**: $\text{AtP}(a) = (a_{\text{corrupt}} - a_{\text{clean}}) \cdot \nabla_a \mathcal{M}(x_{\text{clean}})$
3. **Top-K Sparse Autoencoder**: $z = \text{ReLU}((x - b_{\text{dec}}) W_{\text{enc}} + b_{\text{enc}}), \quad f_z = \text{TopK}(z, k), \quad \hat{x} = f_z W_{\text{dec}} + b_{\text{dec}}$
4. **JumpReLU Activation**: $\text{JumpReLU}(z, \theta) = z \odot \mathbb{I}(z > \theta)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_logit_lens(h, w_u, b_u):
    return h @ w_u + b_u

def calc_attribution_patching(clean, corrupt, grads):
    return ((corrupt - clean) * grads).sum(dim=-1)

def calc_topk_sae(x, w_enc, b_enc, w_dec, b_dec, k=2):
    pre_act = torch.relu((x - b_dec) @ w_enc + b_enc)
    val, idx = torch.topk(pre_act, k=k, dim=-1)
    f_z = torch.zeros_like(pre_act).scatter_(dim=-1, index=idx, src=val)
    return f_z, f_z @ w_dec + b_dec

def calc_jumprelu(z, threshold):
    return z * torch.where(z > threshold, 1.0, 0.0)

In [ ]:
# Parity verification against golden reference tensors
logits = calc_logit_lens(weights["ll_h"], weights["ll_w"], weights["ll_b"])
assert torch.max(torch.abs(logits - weights["ll_logits_ref"])) < 1e-5, "Logit Lens mismatch"

atp = calc_attribution_patching(weights["atp_clean"], weights["atp_corrupt"], weights["atp_grads"])
assert torch.max(torch.abs(atp - weights["atp_scores_ref"])) < 1e-5, "Attribution Patching mismatch"

f_z, x_hat = calc_topk_sae(weights["sae_x"], weights["sae_w_enc"], weights["sae_b_enc"], weights["sae_w_dec"], weights["sae_b_dec"], k=2)
assert torch.max(torch.abs(f_z - weights["sae_feat_ref"])) < 1e-5, "SAE features mismatch"
assert torch.max(torch.abs(x_hat - weights["sae_xhat_ref"])) < 1e-5, "SAE reconstruction mismatch"

jump = calc_jumprelu(weights["jump_z"], weights["jump_thresh"])
assert torch.max(torch.abs(jump - weights["jump_out_ref"])) < 1e-5, "JumpReLU mismatch"
print("All Logit Lens, AtP, Top-K SAE, and JumpReLU assertions passed!")

In [ ]:
# Benchmarking mechanistic circuit analysis operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_logit_lens(weights["ll_h"], weights["ll_w"], weights["ll_b"])
    _ = calc_attribution_patching(weights["atp_clean"], weights["atp_corrupt"], weights["atp_grads"])
    _ = calc_topk_sae(weights["sae_x"], weights["sae_w_enc"], weights["sae_b_enc"], weights["sae_w_dec"], weights["sae_b_dec"], k=2)
    _ = calc_jumprelu(weights["jump_z"], weights["jump_thresh"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused Logit Lens + AtP + SAE latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: L1 Shrinkage Bias in Sparse Dictionaries vs JumpReLU / Top-K
# L1 regularization soft-thresholds active features, artificially dampening true signals
z_true = torch.tensor([5.0, 1.2, 0.3])
l1_lambda = 0.8
z_l1_shrunk = torch.clamp(z_true - l1_lambda, min=0.0)
z_jumprelu = calc_jumprelu(z_true, threshold=torch.tensor(0.5))
print(f"Original: {z_true[0]:.1f} -> L1 Shrunk: {z_l1_shrunk[0]:.1f} vs JumpReLU Unshrunk: {z_jumprelu[0]:.1f}")
assert z_l1_shrunk[0] < z_true[0] and z_jumprelu[0] == z_true[0], "JumpReLU eliminates L1 shrinkage bias!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Logit Lens** decodes intermediate residual activations directly through the model's unembedding head.
2. **Attribution Patching** scales causal mediation analysis to all nodes in a single backward pass.
3. **Top-K & JumpReLU SAEs** unpack polysemantic representations into monosemantic sparse dictionaries without L1 shrinkage bias.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_mechanistic_interpretability_probing_lens_patching_and_sae.md](../../markdown/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.md)
- **PyTorch 2.x Module:** [05_mechanistic_interpretability_probing_lens_patching_and_sae.py](../../pytorch2/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.py)
- **Functional JAX Module:** [05_mechanistic_interpretability_probing_lens_patching_and_sae.py](../../jax/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.py)
- **Native Rust Candle Track:** [05_mechanistic_interpretability_probing_lens_patching_and_sae.rs](../../candle/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.rs)
- **Hardware Kernel (CPU / WGSL):** [05_mechanistic_interpretability_probing_lens_patching_and_sae.rs](../../kernels/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.rs)
- **Golden SafeTensors:** [05_mechanistic_interpretability_probing_lens_patching_and_sae.safetensors](../../weights/10_xai_and_ai_analysis_methods/05_mechanistic_interpretability_probing_lens_patching_and_sae.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. XAI Data Attribution Influence Functions TracIn TRAK](04_xai_data_attribution_influence_functions_tracin_and_trak.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 06. AI Brain Surgery Knowledge Editing ROME MEMIT](06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.ipynb) &rarr;